<a href="https://colab.research.google.com/github/ethnicgarbage/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_23_Jun_Seo_Lee_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [262]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [263]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [264]:
# TODO
print("shape: ", df.shape)
print(" ")
print("types: ", df.dtypes)
print(" ")
print("null values: ", df.isnull().sum())
print(" ")
print("duplicated rows: ", df.duplicated().sum())

shape:  (8, 6)
 
types:  order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
 
null values:  order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
 
duplicated rows:  1


**What is wrong with this data?** List at least five specific problems:

1. Missing item names
2. Inconsistent item names
3. Inconsistent category names
4. Price type as object instead of numerical data
5. Timestamp type as object instead of datetime
6. Invalid/missing data entries
7. Duplicate row(s)

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [265]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
print("removed duplicates: ", removed)
print(" ")

clean =    df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied
print('rows:', len(df), '->', len(clean))
print(" ")

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)
log('duplicates', 'dropped duplicate rows', removed)

removed duplicates:  1
 
rows: 8 -> 7
 
[duplicates] dropped duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [266]:
# TODO: clean['price'] = ...
clean['price'] = (clean['price'].astype(str)
.str.replace("$","", regex=False)
.str.replace(",","", regex=False)
.str.strip()
.astype(float))

assert clean['price'].dtype == float
log('price', 'stripped dollar signs and white spaces, and converted to float', len(clean))

# assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text

[price] stripped dollar signs and white spaces, and converted to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [267]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors = 'coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities


# TODO: apply your decision, then log both separately
clean = clean.loc[clean['qty'].notna()].copy()

log('qty', 'dropped rows with missing quantities', missing)
log('qty', 'retained negative quantities as refunds so revenue reflects returns', negative)

[qty] dropped rows with missing quantities (1 row(s))
[qty] retained negative quantities as refunds so revenue reflects returns (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [268]:
print('before:', sorted(clean['category'].unique()))
before_count = clean['category'].nunique()

# TODO: lowercase, strip, remove punctuation
clean['category'] = (clean['category']
.str.replace("-","", regex=False)
.str.lower()
.str.strip())

# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'apparel' : 'Merch',
    'merch' : 'Merch',
    'food' : 'Food',
    'raingear' : 'RainGear'
}

clean['category'] = clean['category'].replace(CATEGORY_MAP)
after_count = clean['category'].nunique()

print('after: ', sorted(clean['category'].unique()))

log(
    'categories',
    f'normalized labels and mapped Apparel to Merch; '
    f'{before_count} distinct categories became {after_count}',
    len(clean)
)

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Food', 'Merch', 'RainGear']
[categories] normalized labels and mapped Apparel to Merch; 6 distinct categories became 3 (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [269]:
# TODO
print('before:', sorted(clean['item'].dropna().unique()))

missing_items = clean['item'].isna().sum()

clean['item'] = (clean['item']
.str.replace("-","", regex=False)
.str.replace(" ","",regex=False)
.str.lower()
.str.strip())

ITEM_MAP = {
    'cheeseburger' : 'Cheeseburger',
    'foamfinger' : 'Foam Finger',
    'rainponcho' : 'Rain Poncho',
    'uvatshirt' : 'UVA T-Shirt'
}

clean['item'] = clean['item'].replace(ITEM_MAP)
clean['item'] = clean['item'].fillna('Unknown')

print('after: ', sorted(clean['item'].dropna().unique()))

log(
    'items',
    'normalized item spelling, case, whitespace, and punctuation',
    len(clean) - missing_items
)

log(
    'missing item',
    "labeled missing item as 'Unknown' rather than dropping the row",
    missing_items
)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown']
[items] normalized item spelling, case, whitespace, and punctuation (5 row(s))
[missing item] labeled missing item as 'Unknown' rather than dropping the row (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [270]:
# TODO
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')

failed = clean['ts'].isna().sum()

clean['hour'] = clean['ts'].dt.hour

log(
    'timestamps',
    'parsed mixed timestamp formats; kept missing or unparseable values as NaT',
    failed
)

[timestamps] parsed mixed timestamp formats; kept missing or unparseable values as NaT (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [271]:
# TODO: assertions
# 1. Exact duplicates are gone
assert clean.duplicated().sum() == 0

# 2. Price is numeric with no missing values
assert clean['price'].dtype == float
assert clean['price'].notna().all()

# 3. Quantity is numeric and the missing quantity was resolved
assert pd.api.types.is_numeric_dtype(clean['qty'])
assert clean['qty'].notna().all()

# 4. Only the approved categories remain
assert set(clean['category'].unique()) <= {
    'Food', 'Merch', 'RainGear'
}

# 5. Every row has an item label
assert clean['item'].notna().all()
assert 'Unknown' in clean['item'].values

# 6. Timestamp is a real datetime column
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])

# The original blank timestamp should be the only NaT
assert clean['ts'].isna().sum() == 1

# 7. Valid hours must fall between 0 and 23
assert clean['hour'].dropna().between(0, 23).all()


# TODO: clean['revenue'] = ...
clean['revenue'] = clean['qty'] * clean['price']

# TODO: print rows, units, revenue, distinct categories
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print(f"revenue: ${clean['revenue'].sum():.2f}")
print('distinct categories:', clean['category'].nunique())

rows: 6
units: 7.0
revenue: $88.50
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [274]:
show_log()

,step,decision,rows
0,duplicates,dropped duplicate rows,1
1,price,"stripped dollar signs and white spaces, and co...",7
2,qty,dropped rows with missing quantities,1
3,qty,retained negative quantities as refunds so rev...,1
4,categories,normalized labels and mapped Apparel to Merch;...,6
5,items,"normalized item spelling, case, whitespace, an...",5
6,missing item,labeled missing item as 'Unknown' rather than ...,1
7,timestamps,parsed mixed timestamp formats; kept missing o...,1


**The decision that mattered most:** Step 3: retained the -3 quantity as a refund so revenue reflects returns

**Revenue with it:** $88.5

**Revenue without it:** $106.5

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [275]:
# Checkpoint
rows_after = 6            # TODO
revenue_after = 88.5         # TODO
biggest_decision = 'Step 3: retained the -3 quantity as a refund so revenue reflects returns'    # TODO: which choice moved the number most
revenue_other_way = 106.5     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 6
revenue: 88.5
decision that mattered: Step 3: retained the -3 quantity as a refund so revenue reflects returns
revenue the other way: 106.5
